# Topic: Network Monitor Simulator - Packet header parsing, Snort-like signature filters, and security alert triggers
 
## 1. THE THEORY OF PACKET SNIFFING
- **Packet Sniffing / Monitoring**: Capturing raw network traffic passing through network interfaces.
- **Defensive Role**: Intrusion Detection Systems (IDS) scan packet flows to identify signature attacks (like port sweeps, data exfiltration, or database query payloads).
- **CPython Layer**: Raw packet sockets in production environments (like using the `scapy` library or Linux raw sockets `socket(AF_PACKET, SOCK_RAW)`) require administrative root permissions. In this lab, we build a **Network Monitor Simulator** that parses mock network packets, decoding headers and payloads to simulate an active IDS.
 
## 2. SIGNATURE PATTERN MATCHING (IDS Rules)
- **IDS Rules (Snort-style)**:
  - Define filters that inspect protocol type, source/destination IPs, destination ports, and specific text signatures in the packet payload.
  - Example rule: Alert if destination port is `80` and payload contains `"union select"`.
 
---


In [ ]:
import re

class Packet:
    """Simulates a parsed network packet."""
    def __init__(self, src_ip, dest_ip, protocol, dest_port, payload=""):
        self.src_ip = src_ip
        self.dest_ip = dest_ip
        self.protocol = protocol.upper()
        self.dest_port = int(dest_port)
        self.payload = payload

    def __repr__(self):
        return f"Packet({self.src_ip} -> {self.dest_ip} | {self.protocol}:{self.dest_port})"

class SimpleIDS:
    """A lightweight Intrusion Detection System simulator."""
    def __init__(self):
        self.rules = []

    def add_rule(self, rule_name, protocol, dest_port, payload_regex):
        """Adds a signature detection rule to the IDS engine."""
        self.rules.append({
            "name": rule_name,
            "protocol": protocol.upper(),
            "port": int(dest_port),
            "regex": re.compile(payload_regex, re.IGNORECASE)
        })

    def inspect_packet(self, packet):
        """Matches a packet against all registered signature rules."""
        for rule in self.rules:
            # Check protocol and port filters
            if packet.protocol == rule["protocol"] and packet.dest_port == rule["port"]:
                # Check payload signature matches
                if rule["regex"].search(packet.payload):
                    print(f"[ALERT] {rule['name']} MATCHED!")
                    print(f"  Trigger Source: {packet.src_ip} -> {packet.dest_ip}")
                    print(f"  Payload:        {repr(packet.payload)}")
                    return True
        return False

print("--- Initializing Network Monitor Simulator ---")
ids = SimpleIDS()

# Register simple Snort-style rules
ids.add_rule(
    rule_name="SQL Injection attempt in HTTP request",
    protocol="TCP",
    dest_port=80,
    payload_regex=r"(union\s+select|select.*from|or\s+1=1)"
)
ids.add_rule(
    rule_name="SSH Bruteforce payload signature",
    protocol="TCP",
    dest_port=22,
    payload_regex=r"(failed\s+password|invalid\s+user)"
)



In [ ]:
# 2. Simulating Packet Traffic through the Monitor
print("\n--- Simulating Packet Sweeping ---")
packets_stream = [
    Packet("192.168.1.10", "192.168.1.1", "TCP", 80, "GET /index.html HTTP/1.1\r\nHost: local"),
    # Attack packet 1: SQLi in HTTP request
    Packet("10.0.0.5", "192.168.1.1", "TCP", 80, "GET /items?id=1%20union%20select%20null,password%20from%20users"),
    # Normal SSH packet
    Packet("192.168.1.15", "192.168.1.1", "TCP", 22, "SSH-2.0-OpenSSH_8.2"),
    # Attack packet 2: SSH Bruteforce attempts trigger
    Packet("10.0.0.5", "192.168.1.1", "TCP", 22, "Failed password for root from 10.0.0.5 port 4281")
]

alert_count = 0
for i, pkt in enumerate(packets_stream, start=1):
    print(f"\n[Packet {i}] Inspecting: {pkt}")
    if ids.inspect_packet(pkt):
        alert_count += 1

print(f"\nScan finished. Total alerts triggered: {alert_count}")
